# Interoperability with R
Reading Anndata objects stored in .hdf5 format into R is fairly simple. 

In R there are two primary packages used to deal with Single Cell data. SingleCellExperiment objects and Seurat objects.

The following table compares the these.

| Data Component         | AnnData           | SingleCellExperiment               | Seurat                                                                       |
| ---------------------- | ------------------| ---------------------------------- | ---------------------------------------------------------------------------- |
| Count matrix           | `adata.X`         | `assay(sce, "counts")`             | `GetAssayData(seurat, layer = "counts")`                                     |
| Multiple assays/layers | `adata.layers`    | `assays(sce)`                      | `seurat@assays`                                                              |
| Cell metadata          | `adata.obs`       | `colData(sce)`                     | `seurat@meta.data`                                                           |
| Gene metadata          | `adata.var`       | `rowData(sce)`                     | `seurat[["RNA"]]@meta.features`                                              |
| Embeddings             | `adata.obsm`      | `reducedDims(sce)`                 | `Embeddings(seurat, "pca")`, `Embeddings(seurat, "umap")`, etc.              |
| Unstructured metadata  | `adata.uns`       | `metadata(sce)`                    | `seurat@misc`                                                                |

In [ ]:
%load_ext rpy2.ipython

## Reading .hdf5 files into R

In [ ]:
%%R
library(anndataR)
library(scater)

h5ad_path <- "../results/adata/09-annotation.h5ad"

# as `SingleCellExperiment` object 
sce <- read_h5ad(h5ad_path, as = "SingleCellExperiment")

# Or as Seurat object
# seurat <- read_h5ad(h5ad_path, as = "Seurat")

## Exploring the `SingleCellExperiment` object

After reading, the slots from `AnnData` are mapped to their SCE equivalents.

In [ ]:
%%R

# adata.X  →  assay(sce, "X")  (count / normalized matrix)
dim(assay(sce, "X"))       # genes x cells

In [ ]:
%%R

# adata.obs  →  colData(sce)  (cell-level metadata)
head(colData(sce))

In [ ]:
%%R

# adata.var  →  rowData(sce)  (gene-level metadata)
head(rowData(sce))

In [ ]:
%%R

# adata.obsm  →  reducedDims(sce)  (embeddings: PCA, UMAP, …)
reducedDimNames(sce)

## Plotting the UMAP with Scater

In [ ]:
%%R

plotUMAP(sce, colour_by = "leiden_0.5", dimred = "X_umap")

## Writing a `SingleCellExperiment` back to `.h5ad`

`write_h5ad` performs the reverse conversion, producing an `.h5ad` readable by scanpy.

In [ ]:
%%R

write_h5ad(sce, "tmp.h5ad")